**TITLE & IMPORTS**

In [1]:

import pickle
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

print("Imports successful!")

Imports successful!


**SETUP & LOAD MODELS**


In [2]:
PATHS = {'outputs': '../outputs'}

print("Loading models...")
with open(f"{PATHS['outputs']}/nb_model.pkl", 'rb') as f:
    nb_model = pickle.load(f)
with open(f"{PATHS['outputs']}/lr_model.pkl", 'rb') as f:
    lr_model = pickle.load(f)
with open(f"{PATHS['outputs']}/svm_model.pkl", 'rb') as f:
    svm_model = pickle.load(f)
with open(f"{PATHS['outputs']}/vectorizer.pkl", 'rb') as f:
    vectorizer = pickle.load(f)

print("All models loaded!")

Loading models...
All models loaded!


**ENSEMBLE VOTING FUNCTION**

In [3]:
def detect_scam_ensemble(sms_text):
    """Sary 3 models ka voting"""
    X = vectorizer.transform([sms_text])
    
    nb_pred = nb_model.predict(X)[0]
    lr_pred = lr_model.predict(X)[0]
    svm_pred = svm_model.predict(X)[0]
    
    predictions = [nb_pred, lr_pred, svm_pred]
    scam_votes = sum(1 for p in predictions if p == 'Scam')
    
    ensemble_pred = 'Scam' if scam_votes >= 2 else 'Genuine'
    confidence = max(scam_votes, 3 - scam_votes) / 3 * 100
    
    return {
        'prediction': ensemble_pred,
        'confidence': confidence,
        'scam_votes': scam_votes,
        'models': {'NB': nb_pred, 'LR': lr_pred, 'SVM': svm_pred}
    }

print(" Ensemble function created!")

 Ensemble function created!


**PATTERN DETECTION FUNCTION**

In [4]:
def detect_scam_patterns(sms_text):
    """Scam patterns detect karo"""
    patterns = {
        'urgency': ['urgent', 'jaldi', 'immediately', 'alert'],
        'prize': ['mubarak', 'winner', 'lottery', 'prize'],
        'money': ['paise', 'rupees', 'lakh', 'payment'],
        'verification': ['verify', 'otp', 'password', 'confirm'],
        'bank': ['bank', 'account', 'atm', 'debit'],
        'links': ['http', 'www', 'bit.ly', 'click'],
    }
    
    sms_lower = sms_text.lower()
    found_patterns = {}
    risk_score = 0
    
    for pattern_type, keywords in patterns.items():
        matches = [kw for kw in keywords if kw in sms_lower]
        if matches:
            found_patterns[pattern_type] = matches
            risk_score += 20
    
    risk_levels = ['LOW', 'LOW-MEDIUM', 'MEDIUM', 'HIGH', 'CRITICAL']
    risk_level = risk_levels[min(risk_score // 25, 4)]
    
    return {
        'risk_level': risk_level,
        'risk_score': min(risk_score, 100),
        'patterns': found_patterns
    }

print(" Pattern detection function created!")

 Pattern detection function created!


**BATCH PROCESSING FUNCTION**

In [5]:
def batch_detect_scam(sms_list):
    """Multiple SMS process karo"""
    results = []
    
    for sms in sms_list:
        ensemble = detect_scam_ensemble(sms)
        pattern = detect_scam_patterns(sms)
        
        results.append({
            'SMS': sms[:50],
            'Prediction': ensemble['prediction'],
            'Confidence': f"{ensemble['confidence']:.0f}%",
            'Scam_Votes': ensemble['scam_votes'],
            'Risk_Level': pattern['risk_level'],
            'Risk_Score': pattern['risk_score']
        })
    
    return pd.DataFrame(results)

print("Batch processing function created!")

Batch processing function created!


**TEST - SINGLE MESSAGES**

In [6]:
print("\n" + "="*70)
print("TEST 1: SINGLE MESSAGES WITH ALL 3 FEATURES")
print("="*70)

test_messages = [
    "Mubarak ho! Aap lottery winner hain. Prize claim karo",
    "Kya haal hai bhai? Long time no talk",
    "URGENT! Bank account compromised. Verify OTP: www.fake.com",
    "Beta doodh le aana ghar aatay huay"
]

for i, sms in enumerate(test_messages, 1):
    print(f"\n{i}. {sms[:50]}...")
    
    ensemble = detect_scam_ensemble(sms)
    pattern = detect_scam_patterns(sms)
    
    print(f"   Ensemble: {ensemble['prediction']} ({ensemble['confidence']:.0f}%)")
    print(f"   Pattern Risk: {pattern['risk_level']} ({pattern['risk_score']}/100)")
    print(f"   Scam Votes: {ensemble['scam_votes']}/3")


TEST 1: SINGLE MESSAGES WITH ALL 3 FEATURES

1. Mubarak ho! Aap lottery winner hain. Prize claim k...
   Ensemble: Scam (100%)
   Pattern Risk: LOW (20/100)
   Scam Votes: 3/3

2. Kya haal hai bhai? Long time no talk...
   Ensemble: Genuine (100%)
   Pattern Risk: LOW (0/100)
   Scam Votes: 0/3

3. URGENT! Bank account compromised. Verify OTP: www....
   Ensemble: Scam (100%)
   Pattern Risk: HIGH (80/100)
   Scam Votes: 3/3

4. Beta doodh le aana ghar aatay huay...
   Ensemble: Genuine (100%)
   Pattern Risk: LOW (0/100)
   Scam Votes: 0/3


**TEST - BATCH PROCESSING**

In [7]:
print("\n" + "="*70)
print("TEST 2: BATCH PROCESSING")
print("="*70)

df = batch_detect_scam(test_messages)
print("\n" + df.to_string(index=False))

# Statistics
print(f"\n Statistics:")
print(f"Total: {len(df)}")
print(f"Scam: {sum(df['Prediction'] == 'Scam')}")
print(f"Genuine: {sum(df['Prediction'] == 'Genuine')}")


TEST 2: BATCH PROCESSING

                                               SMS Prediction Confidence  Scam_Votes Risk_Level  Risk_Score
Mubarak ho! Aap lottery winner hain. Prize claim k       Scam       100%           3        LOW          20
              Kya haal hai bhai? Long time no talk    Genuine       100%           0        LOW           0
URGENT! Bank account compromised. Verify OTP: www.       Scam       100%           3       HIGH          80
                Beta doodh le aana ghar aatay huay    Genuine       100%           0        LOW           0

 Statistics:
Total: 4
Scam: 2
Genuine: 2


**INTERACTIVE TEST (USER INPUT)**

In [10]:
sms = input(" Enter SMS to check: ")

print("\n" + "="*60)
print("ANALYSIS")
print("="*60)

ensemble = detect_scam_ensemble(sms)
pattern = detect_scam_patterns(sms)

print(f"\n ENSEMBLE VOTING:")
print(f"   Prediction: {ensemble['prediction']}")
print(f"   Confidence: {ensemble['confidence']:.1f}%")
print(f"   Model Votes: {ensemble['models']}")

print(f"\n PATTERN ANALYSIS:")
print(f"   Risk Level: {pattern['risk_level']}")
print(f"   Risk Score: {pattern['risk_score']}/100")
if pattern['patterns']:
    print(f"   Patterns Found: {list(pattern['patterns'].keys())}")
else:
    print(f"   Patterns Found: None")

print(f"\n FINAL VERDICT:")
if ensemble['prediction'] == 'Scam' and pattern['risk_score'] > 50:
    print("    HIGH CONFIDENCE SCAM - BLOCK THIS MESSAGE")
elif ensemble['prediction'] == 'Scam':
    print("     LIKELY SCAM - BE CAREFUL")
elif pattern['risk_score'] > 50:
    print("     MEDIUM RISK - CHECK CAREFULLY")
else:
    print("    LIKELY GENUINE - SAFE TO READ")


ANALYSIS

 ENSEMBLE VOTING:
   Prediction: Genuine
   Confidence: 66.7%
   Model Votes: {'NB': np.str_('Genuine'), 'LR': 'Genuine', 'SVM': 'Scam'}

 PATTERN ANALYSIS:
   Risk Level: LOW
   Risk Score: 0/100
   Patterns Found: None

 FINAL VERDICT:
    LIKELY GENUINE - SAFE TO READ


In [1]:
import pickle
import pandas as pd
import numpy as np

# ═══ LOAD MODELS ═══
PATHS = {'outputs': '../outputs'}

print("Loading models...")
with open(f"{PATHS['outputs']}/nb_model.pkl", 'rb') as f:
    nb_model = pickle.load(f)
with open(f"{PATHS['outputs']}/lr_model.pkl", 'rb') as f:
    lr_model = pickle.load(f)
with open(f"{PATHS['outputs']}/svm_model.pkl", 'rb') as f:
    svm_model = pickle.load(f)
with open(f"{PATHS['outputs']}/vectorizer.pkl", 'rb') as f:
    vectorizer = pickle.load(f)

print("✓ Models loaded!\n")

# ═══ ENSEMBLE FUNCTION ═══
def detect_scam_ensemble(sms_text):
    X = vectorizer.transform([sms_text])
    
    nb_pred = nb_model.predict(X)[0]
    lr_pred = lr_model.predict(X)[0]
    svm_pred = svm_model.predict(X)[0]
    
    predictions = [nb_pred, lr_pred, svm_pred]
    scam_votes = sum(1 for p in predictions if p == 'Scam')
    
    ensemble_pred = 'Scam' if scam_votes >= 2 else 'Genuine'
    confidence = max(scam_votes, 3 - scam_votes) / 3 * 100
    
    return {
        'prediction': ensemble_pred,
        'confidence': confidence,
        'scam_votes': scam_votes
    }

# ═══ PATTERN FUNCTION ═══
def detect_scam_patterns(sms_text):
    patterns = {
        'urgency': ['urgent', 'jaldi', 'immediately', 'alert'],
        'prize': ['mubarak', 'winner', 'lottery', 'prize'],
        'money': ['paise', 'rupees', 'lakh', 'payment'],
        'verification': ['verify', 'otp', 'password', 'confirm'],
        'bank': ['bank', 'account', 'atm', 'debit'],
        'links': ['http', 'www', 'bit.ly', 'click'],
    }
    
    sms_lower = sms_text.lower()
    found_patterns = {}
    risk_score = 0
    
    for pattern_type, keywords in patterns.items():
        matches = [kw for kw in keywords if kw in sms_lower]
        if matches:
            found_patterns[pattern_type] = matches
            risk_score += 20
    
    risk_levels = ['LOW', 'LOW-MEDIUM', 'MEDIUM', 'HIGH', 'CRITICAL']
    risk_level = risk_levels[min(risk_score // 25, 4)]
    
    return {
        'risk_level': risk_level,
        'risk_score': min(risk_score, 100),
        'patterns': found_patterns
    }

print("✓ Functions defined!")

Loading models...
✓ Models loaded!

✓ Functions defined!


In [2]:
completely_new_test = [
    "Salam beta, kaisa chal raha ہے?",
    "URGENT verify account NOW!",
    "Free money - claim here!",
]

print("\n🔍 NEW MESSAGES TEST:\n")

for msg in completely_new_test:
    result = detect_scam_ensemble(msg)
    pattern = detect_scam_patterns(msg)
    
    print(f"📝 {msg}")
    print(f"   Prediction: {result['prediction']}")
    print(f"   Confidence: {result['confidence']:.0f}%")
    print(f"   Risk: {pattern['risk_level']}\n")


🔍 NEW MESSAGES TEST:

📝 Salam beta, kaisa chal raha ہے?
   Prediction: Genuine
   Confidence: 100%
   Risk: LOW

📝 URGENT verify account NOW!
   Prediction: Scam
   Confidence: 100%
   Risk: MEDIUM

📝 Free money - claim here!
   Prediction: Genuine
   Confidence: 100%
   Risk: LOW



In [3]:
"""
Pure Roman Urdu messages test karo
(English nahi, sirf Roman Urdu)
"""

roman_urdu_tests = [
    # Genuine
    "Kya haal ہے? Kaafi time se baat nahi hui",
    "Beta ghar aao, kamaal khana banaya ہے",
    "Meeting kal 2 PM par office mein",
    "Dost, weekend pe movie chalty hain?",
    
    # Scam
    "Mubarak ho! Prize mila - claim karo fori",
    "Urgent! Account verify karo ya block hojayega",
    "Free training + guaranteed job placement",
    "Lakh rupees lottery mein - register karo",
]

print("\n" + "="*70)
print("ROMAN URDU MESSAGES TEST (Pure Urdu/Hinglish)")
print("="*70)

correct = 0
total = 0

for msg in roman_urdu_tests:
    result = detect_scam_ensemble(msg)
    
    # Manual check: kya prediction correct ہے?
    is_scam = any(word in msg.lower() for word in 
                   ['mubarak', 'prize', 'urgent', 'verify', 'free', 'lakh', 'lottery'])
    expected = 'Scam' if is_scam else 'Genuine'
    
    is_correct = result['prediction'] == expected
    correct += is_correct
    total += 1
    
    status = "✓" if is_correct else "❌"
    print(f"\n{status} {msg}")
    print(f"   Prediction: {result['prediction']} (Expected: {expected})")
    print(f"   Confidence: {result['confidence']:.0f}%")

print(f"\n" + "="*70)
print(f"Accuracy: {correct}/{total} = {correct/total*100:.0f}%")
print("="*70)


ROMAN URDU MESSAGES TEST (Pure Urdu/Hinglish)

✓ Kya haal ہے? Kaafi time se baat nahi hui
   Prediction: Genuine (Expected: Genuine)
   Confidence: 67%

✓ Beta ghar aao, kamaal khana banaya ہے
   Prediction: Genuine (Expected: Genuine)
   Confidence: 100%

✓ Meeting kal 2 PM par office mein
   Prediction: Genuine (Expected: Genuine)
   Confidence: 100%

✓ Dost, weekend pe movie chalty hain?
   Prediction: Genuine (Expected: Genuine)
   Confidence: 100%

✓ Mubarak ho! Prize mila - claim karo fori
   Prediction: Scam (Expected: Scam)
   Confidence: 100%

✓ Urgent! Account verify karo ya block hojayega
   Prediction: Scam (Expected: Scam)
   Confidence: 100%

❌ Free training + guaranteed job placement
   Prediction: Genuine (Expected: Scam)
   Confidence: 100%

✓ Lakh rupees lottery mein - register karo
   Prediction: Scam (Expected: Scam)
   Confidence: 100%

Accuracy: 7/8 = 88%
